In [ ]:
import torch
from datasets import load_dataset
import pandas as pd
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
MODEL_PATH =  "/Users/vikaspandey/personal/genesis-lab/data/models/multitask_model_weights.pth"

In [ ]:
conll_data = load_dataset("lhoestq/conll2003")
wiki_data = load_dataset("wikitext", "wikitext-2-raw-v1")

In [ ]:
conll_data["train"]

In [ ]:
def clean_lm_data(texts):
    return [
        text.strip().split() for text in texts if text.strip() and not text.strip().startswith("=")
    ]

def build_vocab(wiki_dataset, conll_dataset, vocab_size=30000):
    word_freq = {}
    idx_to_word = {}
    word_to_idx = {}
    wiki_texts = clean_lm_data(wiki_dataset["text"])
    full_dataset = wiki_texts + list(conll_dataset["tokens"])
    for words in full_dataset:
        for word in words:
            word = word.lower()
            word_freq[word] = word_freq.get(word, 0) + 1
    top_words = sorted(word_freq, key=word_freq.get, reverse=True)[:vocab_size]
    vocab = ["<PAD>", "<UNK>"] + top_words
    idx_to_word = {i: word for i, word in enumerate(vocab)}
    word_to_idx = {word: i for i, word in enumerate(vocab)}
    return vocab, idx_to_word, word_to_idx

def encode_sequence(tokens, word_to_idx):
    unk_idx = word_to_idx["<UNK>"]
    word_indices = [word_to_idx.get(token.lower(), unk_idx) for token in tokens]
    cap_indices = [1 if token and token[0].isupper() else 0 for token in tokens]
    return word_indices, cap_indices

def window_extraction(tokens, word_to_idx, labels, window_size=5):
    half_size = window_size // 2
    tokens_copy = ["<PAD>"]*half_size + tokens + ["<PAD>"]*half_size
    all_word_window = []
    all_cap_window = []
    output_labels = []
    unk_idx = word_to_idx.get("<UNK>")
    for i in range(0, len(tokens)):
        window = tokens_copy[i:(i + window_size)]
        all_word_window.append([word_to_idx.get(token.lower(), unk_idx) for token in window])
        all_cap_window.append([1 if token != "<PAD>" and token[0].isupper() else 0 for token in window])
        output_labels.append(labels[i])
    return torch.tensor(all_word_window), torch.tensor(all_cap_window), torch.tensor(output_labels)

def corrupt_center(word_windows, cap_windows, vocab_size, exclude_words={0, 1}):
    if not isinstance(word_windows, torch.Tensor):
        word_windows = torch.tensor(word_windows)
    if not isinstance(cap_windows, torch.Tensor):
        cap_windows = torch.tensor(cap_windows)

    B, wsz = word_windows.shape
    center = wsz // 2
    min_idx = max(exclude_words) + 1
    
    random_words = torch.randint(min_idx, vocab_size, (B,), device=word_windows.device)
    
    corrupted_words = word_windows.clone()
    corrupted_caps = cap_windows.clone()
    
    corrupted_words[:, center] = random_words
    corrupted_caps[:, center] = 0
    
    return corrupted_words, corrupted_caps

def build_window_loader(dataset, split, word_to_idx, label_field, window_size=5, batch_size=1000, shuffle=True):
    all_words, all_caps, all_labels = [], [], []
    for ex in dataset[split]:
        w, c, l = window_extraction(ex["tokens"], word_to_idx, ex[label_field], window_size)
        all_words.append(w)
        all_caps.append(c)
        all_labels.append(l)
    words = torch.concat(all_words)
    caps = torch.concat(all_caps)
    labels = torch.concat(all_labels)
    ds = TensorDataset(words, caps, labels)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

def build_lm_loader(wiki_dataset, split, word_to_idx, window_size=11, batch_size=1000, shuffle=True):
    all_words, all_caps = [], []
    wiki_texts = clean_lm_data(wiki_dataset[split]["text"])
    for tokens in wiki_texts:
        if len(tokens) < window_size:
            continue
        dummy_label = [0]*len(tokens)
        words, caps, _ = window_extraction(tokens, word_to_idx, dummy_label, window_size=window_size)
        all_words.append(words)
        all_caps.append(caps)
    all_words = torch.concat(all_words)
    all_caps = torch.concat(all_caps)
    ds = TensorDataset(all_words, all_caps)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

def count_classes(dataset, field):
    max_id = 0
    for ex in dataset["train"]:
        if ex[field]:
            max_id = max(max_id, max(ex[field]))
    return max_id + 1

In [ ]:
class SharedEmbedding(nn.Module):
    def __init__(self, vocab_size, wsz, num_cap_features=2, cap_emb_dim=2):
        super().__init__()
        self.word_emb = nn.Embedding(vocab_size, wsz)
        self.cap_emb = nn.Embedding(num_cap_features, cap_emb_dim)
        nn.init.uniform_(self.word_emb.weight, -0.01, 0.01)
        nn.init.uniform_(self.cap_emb.weight, -0.01, 0.01)
    
    def forward(self, word_indices, cap_indices):
        word_embs = self.word_emb(word_indices)   # (B, ksz, wsz)
        cap_embs = self.cap_emb(cap_indices)      # (B, ksz, 2)
        features = torch.cat([word_embs, cap_embs], dim=-1)  # (B, ksz, wsz+2)
        return features

In [ ]:
class WindowClassifier(nn.Module):
    def __init__(self, num_classes, ksz, wsz, cap_emb_dim, hidden_dim=None):
        super().__init__()
        self.num_classes = num_classes
        self.ksz = ksz
        self.wsz = wsz
        self.cap_emb_dim = cap_emb_dim
        self.hidden_dim = hidden_dim
        if hidden_dim is not None:
            self.hidden = nn.Linear(ksz*(wsz + cap_emb_dim), hidden_dim)
            self.output = nn.Linear(hidden_dim, num_classes)
        else:
            self.output = nn.Linear(ksz*(wsz + cap_emb_dim), num_classes)
        for layer in [self.hidden, self.output] if hidden_dim else [self.output]:
            nn.init.uniform_(layer.weight, -0.05, 0.05)
            nn.init.zeros_(layer.bias)
    
    def forward(self, features):
        e = features.reshape(-1, self.ksz*(self.wsz+self.cap_emb_dim))
        if self.hidden_dim is not None:
            h = torch.tanh(self.hidden(e))
        else:
            h = e
        return self.output(h)

In [ ]:
class TDNNClassifier(nn.Module):
    def __init__(self, num_classes, wsz, ksz, cap_emb_dim, n_hu, max_dist, dist_emb_dim, hidden_dim=None):
        super().__init__()
        self.num_classes = num_classes
        self.wsz = wsz
        self.cap_emb_dim = cap_emb_dim
        self.ksz = ksz
        self.n_hu = n_hu
        self.max_dist = max_dist
        self.hidden_dim = hidden_dim
        
        self.dist_emb = nn.Embedding(2 * max_dist + 1, dist_emb_dim)
        self.conv = nn.Conv1d(
            in_channels=wsz + cap_emb_dim + dist_emb_dim,
            out_channels=n_hu,
            kernel_size=ksz,
            padding=(ksz - 1) // 2
        )
        
        if hidden_dim is not None:
            self.hidden = nn.Linear(n_hu, hidden_dim)
            self.output = nn.Linear(hidden_dim, num_classes)
        else:
            self.output = nn.Linear(n_hu, num_classes)
        
        self._init_weights()
    
    def _init_weights(self):
        nn.init.uniform_(self.dist_emb.weight, -0.05, 0.05)
        nn.init.uniform_(self.conv.weight, -0.05, 0.05)
        nn.init.zeros_(self.conv.bias)
        
        layers = [self.output] if self.hidden_dim is None else [self.hidden, self.output]
        for layer in layers:
            nn.init.uniform_(layer.weight, -0.05, 0.05)
            nn.init.zeros_(layer.bias)
            
    def forward(self, features, target_pos):
        B, T, feat_dim = features.shape
        assert feat_dim == self.wsz + self.cap_emb_dim, \
            f"expected feat_dim={self.wsz + self.cap_emb_dim}, got {feat_dim}"
        assert target_pos.shape == (B,), \
            f"expected target_pos shape ({B},), got {target_pos.shape}"
        
        positions = torch.arange(T, device=features.device).unsqueeze(0)
        distances = positions - target_pos.unsqueeze(1)
        assert distances.shape == (B, T)
        
        distances = distances.clamp(-self.max_dist, self.max_dist) + self.max_dist
        dist_emb = self.dist_emb(distances)
        assert dist_emb.shape == (B, T, self.dist_emb.embedding_dim)
        
        features = torch.cat([features, dist_emb], dim=-1)
        assert features.shape == (B, T, self.wsz + self.cap_emb_dim + self.dist_emb.embedding_dim)
        
        features = features.permute(0, 2, 1)
        conv_out = torch.tanh(self.conv(features))
        assert conv_out.shape == (B, self.n_hu, T)  # 'same' padding preserves T
        
        pooled = conv_out.max(dim=2)[0]
        assert pooled.shape == (B, self.n_hu)
        
        if self.hidden_dim is not None:
            pooled = torch.tanh(self.hidden(pooled))
        
        out = self.output(pooled)
        assert out.shape == (B, self.num_classes)
        return out

In [ ]:
class LMScorer(nn.Module):
    def __init__(self, ksz, wsz, cap_emb_dim, hidden_dim):
        super().__init__()
        self.ksz = ksz
        self.wsz = wsz
        self.cap_emb_dim = cap_emb_dim
        self.hidden_dim = hidden_dim

        self.hidden = nn.Linear(ksz*(wsz + cap_emb_dim), hidden_dim)
        self.output = nn.Linear(hidden_dim, 1)

        for layer in [self.hidden, self.output]:
            nn.init.uniform_(layer.weight, -0.05, 0.05)
            nn.init.zeros_(layer.bias)
    
    def forward(self, features):
        B, T, feat_dim = features.shape
        assert T == self.ksz, f"LMScorer expects window of size {self.ksz}, got T={T}"
        assert feat_dim == self.wsz + self.cap_emb_dim, \
            f"Expected feat_dim = {self.wsz + self.cap_emb_dim}, got {feat_dim}"
        e = features.reshape(-1, T*feat_dim)
        h = torch.tanh(self.hidden(e))
        assert h.shape[0] == B, \
            f"Expected num rows in h to be {B} but found {h.shape[0]}"
        assert h.shape[1] == self.hidden_dim, \
            f"Expected num features in h to be {self.hidden_dim} but found {h.shape[1]}"
        output = self.output(h).squeeze(-1)
        assert output.shape == (B,), f"Expected ({B},), got {output.shape}"
        return output

In [ ]:
def step_window_task(model, batch, head, criterion):
    words, caps, labels = batch
    device = next(model.parameters()).device
    words, caps, labels = words.to(device), caps.to(device), labels.to(device)
    features = model.shared_emb(words, caps)
    logits = head(features)
    return criterion(logits, labels)

def step_tdnn_ner(model, batch, criterion):
    words, caps, target_pos, labels = batch 
    device = next(model.parameters()).device
    words, caps, target_pos, labels = words.to(device), caps.to(device), target_pos.to(device), labels.to(device)
    features = model.shared_emb(words, caps)
    logits = model.ner_tdnn_head(features, target_pos)
    return criterion(logits, labels)

def step_lm(model, batch, vocab_size):
    words_pos, caps_pos = batch # dim(word_pos) = b, ksz
    device = next(model.parameters()).device
    words_pos, caps_pos = words_pos.to(device), caps_pos.to(device)
    # dim(word_neg) = b, ksz
    words_neg, caps_neg = corrupt_center(words_pos, caps_pos, vocab_size)
    pos_features = model.shared_emb(words_pos, caps_pos)  # (b, ksz, wsz + num_cap_dim)
    neg_features = model.shared_emb(words_neg, caps_neg)  # (b, ksz, wsz + num_cap_dim)

    f_pos = model.lm_head(pos_features)
    f_neg = model.lm_head(neg_features) 
    return torch.clamp(1 - f_pos + f_neg, min=0).mean()

def evaluate_window_tasks(model, head, data_loader, criterion):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for batch in data_loader:
            loss = step_window_task(model, batch, head, criterion)
            total_loss += loss.item()
        average_loss = total_loss/len(data_loader)
    return average_loss

def evaluate_lm_task(model, data_loader, vocab_size):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for batch in data_loader:
            loss = step_lm(model, batch, vocab_size)
            total_loss += loss.item() if isinstance(loss, torch.Tensor) else loss
        average_loss = total_loss/len(data_loader)
    return average_loss


def evaluate_matrices_helper(model, data_loader, head):
    model.eval()
    with torch.no_grad():
        for batch in data_loader:
            words, caps, labels = batch
            device = next(model.parameters()).device
            words, caps, labels = words.to(device), caps.to(device), labels.to(device)
            features = model.shared_emb(words, caps)
            logits = head(features)
            preds = logits.argmax(dim=-1)
            yield words, preds, labels

def evaluate_pos_accuracy(model, data_loader, head):
    total_correct = 0
    total_size = 0
    for _, preds, actuals in evaluate_matrices_helper(model, data_loader, head):
        total_correct += (preds == actuals).sum().item()
        total_size += preds.numel()
    accuracy = total_correct/total_size
    return accuracy

def evaluate_f1_score(model, data_loader, head, classes):
    num_classes = len(classes)
    confusion = torch.zeros(num_classes, num_classes, dtype=torch.long)
    for _, preds, actuals in evaluate_matrices_helper(model, data_loader, head):
        idx = (actuals*num_classes + preds).cpu()
        confusion += torch.bincount(idx, minlength=num_classes**2).reshape(num_classes, num_classes)
    class_f1_scores = {}
    macro_avg_f1 = 0
    for cls in classes:
        tp = confusion[cls, cls].item()
        fp = confusion[:, cls].sum().item() - tp
        fn = confusion[cls, :].sum().item() - tp
        class_f1_scores[cls] = 2*tp/(2*tp + fp + fn)
        macro_avg_f1 += class_f1_scores[cls]
    return macro_avg_f1/len(classes)

 
def decode_bio_spans(sent_idx, tags):
    """BIO tag sequence -> list of (sent_idx, type, start, end) tuples (end inclusive).

    Lenient on orphan I-X (treated as B-X), matching official conlleval.
    B-X always closes any open span, so adjacent same-type entities stay separate.
    """
    spans = []
    span_type = None
    span_start = None

    def close(end_idx):
        nonlocal span_type, span_start
        if span_type is not None:
            spans.append((sent_idx, span_type, span_start, end_idx))
            span_type, span_start = None, None

    for idx, tag in enumerate(tags):
        if tag.startswith("B-"):
            close(idx - 1)
            span_type, span_start = tag[2:], idx
        elif tag.startswith("I-"):
            # span_type is None  -> orphan I-X, open leniently
            # span_type != tag[2:] -> type change mid-span, close and reopen
            if span_type != tag[2:]:
                close(idx - 1)
                span_type, span_start = tag[2:], idx
        else:  # "O"
            close(idx - 1)

    close(len(tags) - 1)
    return spans


def _precision_recall_f1(gold_spans, pred_spans):
    tp = len(gold_spans & pred_spans)
    fp = len(pred_spans - gold_spans)
    fn = len(gold_spans - pred_spans)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"precision": precision, "recall": recall, "f1": f1, "tp": tp, "fp": fp, "fn": fn}


def evaluate_entity_span_f1_score(ner_test_dataset, model, head, word_to_idx, ksz):
    id_to_ner_tag = {
        0: "O",
        1: "B-PER",
        2: "I-PER",
        3: "B-ORG",
        4: "I-ORG",
        5: "B-LOC",
        6: "I-LOC",
        7: "B-MISC",
        8: "I-MISC"
    }
    device = next(model.parameters()).device
    model.eval()
    all_gold_spans = set()
    all_pred_spans = set()

    with torch.no_grad():
        for sent_idx, test_data in enumerate(ner_test_dataset):
            tokens = test_data["tokens"]
            gold_ids = test_data["ner_tags"]
            if not tokens:
                continue
            words, caps, _ = window_extraction(tokens, word_to_idx, gold_ids, ksz)
            words, caps = words.to(device), caps.to(device)
            features = model.shared_emb(words, caps)
            logits = head(features)
            # .tolist() is required: iterating a tensor yields 0-dim tensors,
            # which are hashed by identity and would silently miss every dict key
            pred_ids = logits.argmax(dim=-1).tolist()
            pred_tags = [id_to_ner_tag[i] for i in pred_ids]
            gold_tags = [id_to_ner_tag[i] for i in gold_ids]
            assert len(pred_tags) == len(gold_tags) == len(tokens), (
                f"tag/token length mismatch at sentence {sent_idx}: "
                f"{len(pred_tags)} preds, {len(gold_tags)} gold, {len(tokens)} tokens"
            )
            all_gold_spans.update(decode_bio_spans(sent_idx, gold_tags))
            all_pred_spans.update(decode_bio_spans(sent_idx, pred_tags))

    overall = _precision_recall_f1(all_gold_spans, all_pred_spans)
    assert overall["tp"] + overall["fp"] == len(all_pred_spans)
    assert overall["tp"] + overall["fn"] == len(all_gold_spans)

    per_type = {}
    for entity_type in sorted({span[1] for span in all_gold_spans | all_pred_spans}):
        gold_t = {s for s in all_gold_spans if s[1] == entity_type}
        pred_t = {s for s in all_pred_spans if s[1] == entity_type}
        per_type[entity_type] = _precision_recall_f1(gold_t, pred_t)

    return {"overall": overall, "per_type": per_type}


def print_span_f1_report(results):
    overall = results["overall"]
    print(f"{'type':<8} {'P':>7} {'R':>7} {'F1':>7} {'TP':>6} {'FP':>6} {'FN':>6}")
    print("-" * 52)
    for entity_type, m in results["per_type"].items():
        print(f"{entity_type:<8} {m['precision']:>7.4f} {m['recall']:>7.4f} "
              f"{m['f1']:>7.4f} {m['tp']:>6} {m['fp']:>6} {m['fn']:>6}")
    print("-" * 52)
    print(f"{'MICRO':<8} {overall['precision']:>7.4f} {overall['recall']:>7.4f} "
          f"{overall['f1']:>7.4f} {overall['tp']:>6} {overall['fp']:>6} {overall['fn']:>6}")



In [ ]:
class MultitaskModel(nn.Module):
    def __init__(self, shared_emb, pos_head, ner_window_head, ner_tdnn_head, chunk_head, lm_head):
        super().__init__()
        self.shared_emb = shared_emb
        self.pos_head = pos_head
        self.ner_window_head = ner_window_head
        self.ner_tdnn_head = ner_tdnn_head
        self.chunk_head = chunk_head
        self.lm_head = lm_head

In [ ]:
vocab_size = 30000
vocab, idx_to_word, word_to_idx = build_vocab(wiki_data["train"], conll_data["train"], vocab_size = vocab_size)
vocab_size = len(vocab)
features = conll_data["train"].features
num_steps = 5000
embedding_dims = 50
supervised_wndow_size = 5
lm_window_size = 11
num_cap_features = 2
cap_emb_dim = 2
chunking_nhu = 200
lm_nhu = 100
pos_num_classes = count_classes(conll_data, "pos_tags")
chunk_num_classes = count_classes(conll_data, "chunk_tags")
ner_num_classes = count_classes(conll_data, "ner_tags")
shared_emb = SharedEmbedding(vocab_size, embedding_dims, num_cap_features, cap_emb_dim)
pos_head = WindowClassifier(pos_num_classes, supervised_wndow_size, embedding_dims, cap_emb_dim=cap_emb_dim)
ner_head = WindowClassifier(ner_num_classes, supervised_wndow_size, embedding_dims, cap_emb_dim=cap_emb_dim)
chunk_head = WindowClassifier(chunk_num_classes, supervised_wndow_size, embedding_dims, cap_emb_dim=cap_emb_dim, hidden_dim=chunking_nhu)
lm_head = LMScorer(lm_window_size, embedding_dims, cap_emb_dim, hidden_dim=lm_window_size)

device = torch.device('cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu'))
print(f"Using device: {device}")
model = MultitaskModel(shared_emb, pos_head, ner_head, None, chunk_head, lm_head).to(device)

In [ ]:
def load_model(model):
    state_dict = torch.load(MODEL_PATH, map_location=device)
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    return model

In [ ]:
import random
pos_loader = build_window_loader(conll_data, "train", word_to_idx, "pos_tags")
ner_loader = build_window_loader(conll_data, "train", word_to_idx, "ner_tags")
chunk_loader = build_window_loader(conll_data, "train", word_to_idx, "chunk_tags")
lm_loader = build_lm_loader(wiki_data, "train", word_to_idx)
pos_loader_eval = build_window_loader(conll_data, "validation", word_to_idx, "pos_tags")
ner_loader_eval = build_window_loader(conll_data, "validation", word_to_idx, "ner_tags")
chunk_loader_eval = build_window_loader(conll_data, "validation", word_to_idx, "chunk_tags")
pos_loadet_test = build_window_loader(conll_data, "test", word_to_idx, "pos_tags")
ner_test_loader = build_window_loader(conll_data, "test", word_to_idx, "ner_tags")
chunk_test_loader = build_window_loader(conll_data, "test", word_to_idx, "chunk_tags")
lm_loader_eval = build_lm_loader(wiki_data, "validation", word_to_idx)
num_steps = 5000

print("Dataset Loader Lengths (Number of batches):")
print(f"POS   - Train: {len(pos_loader)}, Eval: {len(pos_loader_eval)}")
print(f"NER   - Train: {len(ner_loader)}, Eval: {len(ner_loader_eval)}")
print(f"Chunk - Train: {len(chunk_loader)}, Eval: {len(chunk_loader_eval)}")
print(f"LM    - Train: {len(lm_loader)}, Eval: {len(lm_loader_eval)}")

learning_rate = 0.001
task_loader_head_map = {
    "pos": (pos_loader, model.pos_head, pos_loader_eval),
    "ner_w": (ner_loader, model.ner_window_head, ner_loader_eval),
    "chunk": (chunk_loader, model.chunk_head, chunk_loader_eval),
    "lm": (lm_loader, model.lm_head, lm_loader_eval)
}
iters = {task: iter(values[0]) for task, values in task_loader_head_map.items()}
losses = {task: [] for task in task_loader_head_map.keys()}

def get_batch(task):
    try:
        return next(iters[task])
    except StopIteration:
        iters[task] = iter(task_loader_head_map[task][0])
        return next(iters[task])

def train():
    tasks = list(task_loader_head_map.keys())
    ce_loss = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    for step in range(num_steps):
        model.train()
        task = random.choice(tasks)
        batch = get_batch(task)
        if task in ["pos", "ner_w", "chunk"]:
            loss = step_window_task(model, batch, task_loader_head_map[task][1], ce_loss)
        if task == "lm":
            loss = step_lm(model, batch, vocab_size)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if task in ["pos", "ner_w", "chunk"]:
            eval_loss = evaluate_window_tasks(model, task_loader_head_map[task][1], task_loader_head_map[task][2], ce_loss)
        if task == "lm":
            eval_loss = evaluate_lm_task(model, task_loader_head_map[task][2], vocab_size)
        losses[task].append((loss.item(), eval_loss.item() if isinstance(eval_loss, torch.Tensor) else eval_loss))
        if step % 200 == 0:
            print(f"step={step:5d} task={task:6s} loss={loss.item():.4f} eval_loss={eval_loss:.4f}")

In [ ]:
import matplotlib.pyplot as plt

def plot_task_losses(losses):
    """
    Plots the training and validation losses for each task in a 2x2 grid.
    """
    tasks = list(losses.keys())
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    axes = axes.flatten()
    
    for idx, task in enumerate(tasks):
        ax = axes[idx]
        task_losses = losses[task]
        if not task_losses:
            ax.text(0.5, 0.5, f"No data for {task}", 
                    horizontalalignment='center', verticalalignment='center', fontsize=12)
            ax.set_title(f"Task: {task}")
            continue
            
        steps = list(range(1, len(task_losses) + 1))
        train_losses = [t for t, v in task_losses]
        eval_losses = [v for t, v in task_losses]
        
        ax.plot(steps, train_losses, label='Training Loss', color='#1f77b4', linewidth=2)
        ax.plot(steps, eval_losses, label='Validation Loss', color='#ff7f0e', linewidth=2, linestyle='--')
        
        ax.set_title(f"Task: {task.upper()}", fontsize=14, fontweight='bold')
        ax.set_xlabel("Updates/Steps", fontsize=12)
        ax.set_ylabel("Loss", fontsize=12)
        ax.legend(fontsize=10)
        ax.grid(True, linestyle=':', alpha=0.6)
        
    plt.tight_layout()
    plt.show()

# Call the function to plot
plot_task_losses(losses)

In [ ]:
torch.save(
    model.state_dict(), 
    MODEL_PATH
)

In [ ]:
loaded_model = load_model(model)
ce_loss = nn.CrossEntropyLoss()
pos_eval_losses = evaluate_window_tasks(loaded_model, pos_head, pos_loader_eval, ce_loss)
print(pos_eval_losses)

In [ ]:
evaluate_pos_accuracy(loaded_model, pos_loadet_test, pos_head)

In [ ]:
conll_data_train = conll_data["train"]
conll_data_train_df = conll_data["train"].to_pandas()
conll_data_train_df = conll_data_train_df.explode(["tokens", "pos_tags", "chunk_tags", "ner_tags"], ignore_index=False)
conll_data_train_df.head()
ner_tags_distribution = (
    conll_data_train_df.groupby("ner_tags")
    .agg(
        total_count=("tokens", "count"), 
        percentage_count=("tokens", lambda tokens: tokens.count()/conll_data_train_df.shape[0])
    )
    .reset_index()
)
ner_tags_distribution.head()
evaluate_f1_score(loaded_model, ner_test_loader, ner_head, list(ner_tags_distribution["ner_tags"]))
result = evaluate_entity_span_f1_score(conll_data["test"], loaded_model, ner_head, word_to_idx, 5)
print_span_f1_report(result)